# Reading output from part 2

In [1]:
import pandas as pd
from pathlib import Path

json_path = Path('/kaggle/input/notebooks/sebastianvpal/improving-baseline-model-part2/hp_search_results_part2.jsonl')
json_results = pd.read_json(json_path,lines = True)
# Sorting the results: from best to worst
json_results = json_results.sort_values("best_score", ascending = False)

In [2]:
json_results.head(10)


,trial_id,elapsed_s,fold,n_epochs,max_iters,seed,data_parallel,batch_size,method,lr,det_loss_weight,det_neg_weight,det_threshold,best_score,final_edge_loss,final_det_loss,final_test_acc,final_test_recall,best_test_acc,best_test_recall
3,tier1_003,1006.2,0,4,200,0,False,2,hp_search_tier1,0.00005,4,0.010,0.80,0.925941,0.000640,0.006326,0.998590,0.887633,0.998633,0.927208
0,tier1_000,1022.8,0,4,200,0,False,2,hp_search_tier1,0.00001,4,0.030,0.80,0.917582,0.000659,0.016699,0.998729,0.898940,0.998753,0.918728
1,tier1_001,1010.8,0,4,200,0,False,2,hp_search_tier1,0.00003,4,0.010,0.80,0.916434,0.000581,0.006555,0.998894,0.894700,0.999040,0.917314
7,tier1_007,1006.8,0,4,200,0,False,2,hp_search_tier1,0.00003,3,0.030,0.99,0.911712,0.000651,0.014730,0.998508,0.913074,0.998508,0.913074
2,tier1_002,1007.4,0,4,200,0,False,2,hp_search_tier1,0.00001,4,0.030,0.99,0.909153,0.000733,0.017092,0.998798,0.910247,0.998798,0.910247
5,tier1_005,1009.2,0,4,200,0,False,2,hp_search_tier1,0.00005,4,0.010,0.92,0.906162,0.000664,0.006762,0.998613,0.907420,0.998613,0.907420
6,tier1_006,1014.6,0,4,200,0,False,2,hp_search_tier1,0.00003,2,0.003,0.99,0.905376,0.000511,0.002632,0.999126,0.841696,0.999303,0.906007
8,tier1_008,1021.0,0,4,200,0,False,2,hp_search_tier1,0.00003,3,0.003,0.92,0.901002,0.000509,0.003354,0.999021,0.895406,0.999152,0.901767
11,tier1_011,1021.9,0,4,200,0,False,2,hp_search_tier1,0.00001,3,0.003,0.99,0.900369,0.000502,0.002734,0.999233,0.901060,0.999233,0.901060
9,tier1_009,1013.6,0,4,200,0,False,2,hp_search_tier1,0.00005,2,0.030,0.80,0.899984,0.000688,0.014352,0.998715,0.857951,0.998806,0.901060


In [3]:
# sanity-check marginal effect of each param independently
to_chek = ["lr", "det_loss_weight", "det_neg_weight", "det_threshold"]
for param in [*to_chek]:
    print(json_results.groupby(param)["best_score"].agg(["mean", "std", "count"]).sort_values("mean", ascending = False),"\n","--"*20)

             mean       std  count
lr                                
0.00005  0.910696  0.013560      3
0.00003  0.908631  0.006811      4
0.00001  0.901675  0.012126      5 
 ----------------------------------------
                     mean       std  count
det_loss_weight                           
4                0.915054  0.007758      5
3                0.901955  0.007088      4
2                0.897297  0.009706      3 
 ----------------------------------------
                    mean       std  count
det_neg_weight                           
0.010           0.916179  0.009892      3
0.030           0.909608  0.007322      4
0.003           0.897603  0.007253      5 
 ----------------------------------------
                   mean       std  count
det_threshold                           
0.80           0.910936  0.013056      5
0.99           0.906652  0.004932      4
0.92           0.897898  0.010177      3 
 ----------------------------------------


In [4]:
json_results.groupby("det_threshold")[["best_score", "best_test_acc", "best_test_recall"]].agg(["mean", "std"])

best_score           best_test_acc           best_test_recall  \
                    mean       std          mean       std             mean   
det_threshold                                                                 
0.80            0.910936  0.013056      0.998897  0.000248         0.911943   
0.92            0.897898  0.010177      0.998841  0.000279         0.898940   
0.99            0.906652  0.004932      0.998961  0.000376         0.907597   

                         
                    std  
det_threshold            
0.80           0.013227  
0.92           0.010192  
0.99           0.005237

**Observations:**

- The `learning_rate`optimal value was reached with 5e-5. However, other value candidates as 3e-5 and 1e-5 reched good values as well. An idea is to implement a `callback`using this information. Stating at a large value such as 1e-4 and decreasing util 1e-5.
- The optimal value for `det_loss_weight`is around 4 ( with 4 and 3 the best score reached similar values)
- In the previous notebook (part2) the best value for `det_neg_weight` was 0.003 with 0.9081 followed by 0.030 with 0.9070. In this ocassion the best score was obtained with 0.01 (0.9161 best score) and followd by 0.03 (0.9096) this migh indicate that the best value  for this metric is in the order of 1e-2 to 3e-2 and not 3e-3.
- As in part2 of this search, the optimal value for `det_threshold` was obtained with 0.80, moving to a larger value such as 0.92 didn't give any benefit or improvement in the metric.

---

**Observation 2:**

The values from `final_test_acc`, `final_test_recall`, `best_test_acc`, `best_test_recall` from this notebook and part2 shown that the model is struggling to accuratly identify nodes (cells) while with the identified edges (links between cells) are correctly predicted. This is supported by looking at `best_test_acc`, `best_test_recall`, for example, `best_test_acc` is related to edge detection and it achieves a value of ~ 0.99 while `best_test_recall` (associated to node detection) gets ~ 0.91, meaning that 9% (in average, for this short test) of the nodes are wrong. This is important as **we can isolate where the model is performing worst and plan strategies to improve this aspect.**

---
**caveat:**

Statistica significance. Every analyzed group has 3-5 samples with std values ranging from 0.005 to 0.014 with  group means in the same order of magnitude, making weaker the conclusion drawn from difference in the means between groups, i.e. many of these group differences are within about one standard error of each other. Nevertheless, it does not mean that these results are meaningless but it is important to take into account when defining values for a larger training. 

---
**Conclusion from Set 1 analysis**

From these two searches it is possible to conclude that: the best value for `learning rate`lies between 9e-4 and 3e-5, for this reason a `Callback` will be implemented. Optimal values for `det_loss_weight`, `det_neg_weight` and `det_threshold` are **4, 1e-2, and 0.80, respectively**.

# Sweep on set 2.

Now we are going to move one and perform the search for the optimal values for the set of variables we have defined as "set 2" (check first part of these notebook series). The fist steps of the notebook are the same as for the previous ones, i.e. configuration, setting environment, code modifications, the only change is in the variables sweep

## Defining input data

In [5]:
from pathlib import Path
import os
DATA_PATH = Path('/kaggle/input/competitions/biohub-cell-tracking-during-development')
TRAINING_PATH =  DATA_PATH/'train'
TEST_PATH = DATA_PATH/'test'


#------------ Listing folders in train -----------
print("Input contents:", os.listdir(TRAINING_PATH)[:5],"\n")
print(" First 5 elements in train folder, it contains '.geff' folders (annotated graphs) and '.zarr' folder (videos)")

Input contents: ['6bba_2540cd90.geff', '44b6_0b24845f.geff', '44b6_996155de.geff', '44b6_0c582fdc.geff', '6bba_cf35214c.zarr'] 

 First 5 elements in train folder, it contains '.geff' folders (annotated graphs) and '.zarr' folder (videos)


## Modifications to baseline model

In [6]:
import shutil, sys
from pathlib import Path
# --------- Making a copy of the folder to my own working space --------------
# ---------             such that I can edit it     --------------------------

ARTIFACTS_SRC  = Path("/kaggle/input/datasets/thibautgoldsborough/cellmot-baseline-artifacts/cellmot-baseline-artifacts")
ARTIFACTS_WORK = Path("/kaggle/working/cellmot-baseline-artifacts")

if not ARTIFACTS_WORK.exists():
    shutil.copytree(ARTIFACTS_SRC, ARTIFACTS_WORK)

# put the writable copy ahead of anything else on the path
sys.path.insert(0, str(ARTIFACTS_WORK))
sys.path.insert(0, str(ARTIFACTS_WORK / "repo/scripts"))   # adjust to wherever train_unet_transformer.py actually sits
sys.path.insert(0, str(ARTIFACTS_WORK / "repo/src"))

In [7]:
needed_dirs = {
    matches[0].parent for name in
    ("train_unet_transformer.py", "tracking_cellmot", "augmentations.py", "dataspec.py")
    if (matches := list(ARTIFACTS_WORK.rglob(name)))
}
needed_dirs

{PosixPath('/kaggle/working/cellmot-baseline-artifacts/repo/scripts'),
 PosixPath('/kaggle/working/cellmot-baseline-artifacts/repo/src')}

In [8]:
import glob
import os
import subprocess
import sys
import shutil

# 1. Find all wheels, but filter OUT numpy wheels to avoid breaking C-extensions
wheels = [
    f for f in glob.glob(f"{ARTIFACTS_SRC}/wheels/*.whl")
    if "numpy" not in os.path.basename(f).lower()
]

# 2. Install only the required non-NumPy wheels without upgrading dependencies
subprocess.run(
    [
        "pip", "install", 
        "--no-index", 
        "--find-links", f"{ARTIFACTS_SRC}/wheels",
        "--no-deps",
        *wheels
    ],
    check=True)

Looking in links: /kaggle/input/datasets/thibautgoldsborough/cellmot-baseline-artifacts/cellmot-baseline-artifacts/wheels
Processing /kaggle/input/datasets/thibautgoldsborough/cellmot-baseline-artifacts/cellmot-baseline-artifacts/wheels/imagecodecs-2026.6.26-cp312-abi3-manylinux_2_28_x86_64.whl
Processing /kaggle/input/datasets/thibautgoldsborough/cellmot-baseline-artifacts/cellmot-baseline-artifacts/wheels/shellingham-1.5.4-py2.py3-none-any.whl
Processing /kaggle/input/datasets/thibautgoldsborough/cellmot-baseline-artifacts/cellmot-baseline-artifacts/wheels/numcodecs-0.15.1-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl
Processing /kaggle/input/datasets/thibautgoldsborough/cellmot-baseline-artifacts/cellmot-baseline-artifacts/wheels/bidict-0.23.1-py3-none-any.whl
Processing /kaggle/input/datasets/thibautgoldsborough/cellmot-baseline-artifacts/cellmot-baseline-artifacts/wheels/requests-2.34.2-py3-none-any.whl
Processing /kaggle/input/datasets/thibautgoldsborough/cellmot-bas

CompletedProcess(args=['pip', 'install', '--no-index', '--find-links', '/kaggle/input/datasets/thibautgoldsborough/cellmot-baseline-artifacts/cellmot-baseline-artifacts/wheels', '--no-deps', '/kaggle/input/datasets/thibautgoldsborough/cellmot-baseline-artifacts/cellmot-baseline-artifacts/wheels/imagecodecs-2026.6.26-cp312-abi3-manylinux_2_28_x86_64.whl', '/kaggle/input/datasets/thibautgoldsborough/cellmot-baseline-artifacts/cellmot-baseline-artifacts/wheels/shellingham-1.5.4-py2.py3-none-any.whl', '/kaggle/input/datasets/thibautgoldsborough/cellmot-baseline-artifacts/cellmot-baseline-artifacts/wheels/numcodecs-0.15.1-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl', '/kaggle/input/datasets/thibautgoldsborough/cellmot-baseline-artifacts/cellmot-baseline-artifacts/wheels/bidict-0.23.1-py3-none-any.whl', '/kaggle/input/datasets/thibautgoldsborough/cellmot-baseline-artifacts/cellmot-baseline-artifacts/wheels/requests-2.34.2-py3-none-any.whl', '/kaggle/input/datasets/thibautgolds

**Creating splits_filefile for the training and test datasets**


In [9]:
import json
import random
from pathlib import Path

if not TRAINING_PATH.is_dir():
    raise FileNotFoundError(f"TRAINING_PATH does not exist or is not a directory: {TRAINING_PATH}")

# stems with matching .geff annotation, same convention as the rest of the pipeline
train_pool_stems = sorted(
    p.stem for p in TRAINING_PATH.iterdir()
    if p.is_dir() and p.suffix == ".zarr"
    and (TRAINING_PATH / f"{p.stem}.geff").exists()
)
print(f"{len(train_pool_stems)} videos available in training pool")


MAX_SAMPLES = 10 # define how many datasets will be considered MAX_SAMPLES <= len(train_pool_stems)
VAL_FRACTION = 0.15  # 15% of the total considered data 
n_val = max(1, round(MAX_SAMPLES * VAL_FRACTION))

rng = random.Random(0)          # fixed seed — same split reused across every hp-search trial
shuffled = train_pool_stems.copy()
rng.shuffle(shuffled)
subset = shuffled[:MAX_SAMPLES]

val_stems = sorted(subset[:n_val])
train_stems = sorted(subset[n_val:])

print(f"{len(train_stems)} train / {len(val_stems)} val")
assert set(train_stems).isdisjoint(val_stems)  # sanity check — no leakage between the two

splits = [{"split": 0, "train": train_stems, "test": val_stems}]
# if split = "all" the "train()" function performs 5 folds during training
with open(f"{ARTIFACTS_WORK}/kaggle_train_val_splits.json", "w") as f:
    json.dump(splits, f, indent=2)

199 videos available in training pool
8 train / 2 val


In [10]:
#---------- Reading back the json file just created ------------
with open(f"{ARTIFACTS_WORK}/kaggle_train_val_splits.json", "r") as f:
    json_file = json.load(f)

json_file

[{'split': 0,
  'train': ['44b6_1574802b',
   '44b6_d5e7d891',
   '6bba_2312ac41',
   '6bba_5c824876',
   '6bba_7af54fde',
   '6bba_7b5d3b2c',
   '6bba_afb141ff',
   '6bba_d1acb6ff'],
  'test': ['44b6_d754aa59', '6bba_268e1230']}]

Applying modifications to the baseline code


Working now on the folder located in my work folder

In [11]:
REPO_DIR = ARTIFACTS_WORK / "repo"
target = REPO_DIR / "scripts" / "train_unet_transformer.py"  # path to the training script
src = target.read_text()

# 1. thread det_threshold through train_epoch
src = src.replace(
    "def train_epoch(\n    model: UNetNodeTransformer,\n    loader: DataLoader,\n    optimizer: torch.optim.Optimizer,\n    device: torch.device,\n    det_loss_weight: float = 0.1,\n    det_neg_weight: float = 0.1,\n    max_iters: int | None = None,\n    pool_kernel_um: float = 5.0,\n) -> tuple[float, float]:",
    "def train_epoch(\n    model: UNetNodeTransformer,\n    loader: DataLoader,\n    optimizer: torch.optim.Optimizer,\n    device: torch.device,\n    det_loss_weight: float = 0.1,\n    det_neg_weight: float = 0.1,\n    max_iters: int | None = None,\n    pool_kernel_um: float = 5.0,\n    det_threshold: float = 0.3,\n) -> tuple[float, float]:"
)
src = src.replace(
    "                voxel_size=voxel_size,\n                pool_kernel_um=pool_kernel_um,\n                frame_index=i, window_size=W,\n            )\n            unet_feat = model._index_features(\n                unet_out[:, i], det_c, det_m,\n            )\n            frame_det.append((det_c, det_p, det_m, matches, unet_feat))\n\n        # --- 4. Per-pair edge prediction",
    "                voxel_size=voxel_size,\n                pool_kernel_um=pool_kernel_um,\n                det_threshold=det_threshold,\n                frame_index=i, window_size=W,\n            )\n            unet_feat = model._index_features(\n                unet_out[:, i], det_c, det_m,\n            )\n            frame_det.append((det_c, det_p, det_m, matches, unet_feat))\n\n        # --- 4. Per-pair edge prediction"
)

# 2. train() returns metrics instead of just the model
src = src.replace(
    "    print(f\"\\nBest score (acc*recall): {best_score:.4f}, saved to {save_path}\")\n    if save_path.exists():",
    "    print(f\"\\nBest score (acc*recall): {best_score:.4f}, saved to {save_path}\")\n    metrics = {\"best_score\": best_score, \"final_edge_loss\": edge_loss,\n               \"final_det_loss\": det_loss, \"final_test_acc\": test_acc,\n               \"final_test_recall\": test_recall}\n    if save_path.exists():"
)
src = src.replace(
    "        model.load_state_dict(state)\n    return model",
    "        model.load_state_dict(state)\n    return model, metrics"
)

target.write_text(src)

49486

In [12]:
REPO_DIR = ARTIFACTS_WORK / "repo"

target = REPO_DIR / "scripts" / "train_unet_transformer.py"  # same path as before
src = target.read_text()

# 1. add full_checkpoint to train()'s signature
src = src.replace(
    "    pool_kernel_um: float = 5.0,\n    data_parallel: bool = True,\n) -> UNetNodeTransformer:",
    "    pool_kernel_um: float = 5.0,\n    data_parallel: bool = True,\n    full_checkpoint: Path | None = None,\n    det_threshold: float = 0.3,\n) -> UNetNodeTransformer:"
)

# 2. load it right after model construction, BEFORE any DataParallel wrapping
#    (checkpoint keys are unwrapped "unet.*", wrapping would change them to "unet.module.*")
src = src.replace(
    "    model = UNetNodeTransformer(\n        unet=unet,\n        unet_out_channels=unet_out_channels,\n        pos_feat_dim=pos_feat_dim,\n    ).to(device)\n",
    "    model = UNetNodeTransformer(\n        unet=unet,\n        unet_out_channels=unet_out_channels,\n        pos_feat_dim=pos_feat_dim,\n    ).to(device)\n\n"
    "    if full_checkpoint is not None:\n"
    "        ckpt_state = torch.load(full_checkpoint, map_location=device, weights_only=True)\n"
    "        missing, unexpected = model.load_state_dict(ckpt_state, strict=False)\n"
    "        print(f\"  Full checkpoint loaded from {full_checkpoint}: \"\n"
    "              f\"{len(missing)} missing, {len(unexpected)} unexpected\", flush=True)\n"
    "        if missing or unexpected:\n"
    "            print(f\"    missing (sample): {missing[:5]}\", flush=True)\n"
    "            print(f\"    unexpected (sample): {unexpected[:5]}\", flush=True)\n"
)

# 2. pass it into the train_epoch(...) call inside the epoch loop
src = src.replace(
    "        edge_loss, det_loss = train_epoch(\n"
    "            model, train_loader, optimizer, device, det_loss_weight, det_neg_weight,\n"
    "            max_iters=max_iters, pool_kernel_um=pool_kernel_um,\n"
    "        )",
    "        edge_loss, det_loss = train_epoch(\n"
    "            model, train_loader, optimizer, device, det_loss_weight, det_neg_weight,\n"
    "            max_iters=max_iters, pool_kernel_um=pool_kernel_um, det_threshold=det_threshold,\n"
    "        )"
)

# 3. pass it into the evaluate(...) call inside the epoch loop
src = src.replace(
    "        test_loss, test_acc, test_recall = evaluate(model, test_loader, device, pool_kernel_um=pool_kernel_um)",
    "        test_loss, test_acc, test_recall = evaluate(model, test_loader, device, pool_kernel_um=pool_kernel_um, det_threshold=det_threshold)"
)


target.write_text(src)

# sanity checks — confirm both edits actually matched before trusting them
assert "full_checkpoint: Path | None = None," in src
assert "Full checkpoint loaded from" in src
assert "det_threshold: float = 0.3,\n) -> UNetNodeTransformer:" in src
assert "pool_kernel_um=pool_kernel_um, det_threshold=det_threshold,\n        )" in src
assert "evaluate(model, test_loader, device, pool_kernel_um=pool_kernel_um, det_threshold=det_threshold)" in src

In [13]:
# 1. add det_threshold to evaluate()'s signature
src = src.replace(
    "def evaluate(\n"
    "    model: UNetNodeTransformer,\n"
    "    loader: DataLoader,\n"
    "    device: torch.device,\n"
    "    pool_kernel_um: float = 5.0,\n"
    ") -> tuple[float, float, float]:",
    "def evaluate(\n"
    "    model: UNetNodeTransformer,\n"
    "    loader: DataLoader,\n"
    "    device: torch.device,\n"
    "    pool_kernel_um: float = 5.0,\n"
    "    det_threshold: float = 0.3,\n"
    ") -> tuple[float, float, float]:"
)

# 2. pass it into detect_and_match(...) inside evaluate()'s loop
src = src.replace(
    "            det_c, det_p, det_m, matches = detect_and_match(\n"
    "                det_logits[i], coords[:, i], masks[:, i],\n"
    "                image_shape,\n"
    "                voxel_size=voxel_size,\n"
    "                pool_kernel_um=pool_kernel_um,\n"
    "                frame_index=i, window_size=W,\n"
    "            )\n"
    "            unet_feat = model._index_features(\n"
    "                unet_out[:, i], det_c, det_m,\n"
    "            )\n"
    "            frame_det.append((det_c, det_p, det_m, matches, unet_feat))\n"
    "\n"
    "            # Node recall:",
    "            det_c, det_p, det_m, matches = detect_and_match(\n"
    "                det_logits[i], coords[:, i], masks[:, i],\n"
    "                image_shape,\n"
    "                voxel_size=voxel_size,\n"
    "                pool_kernel_um=pool_kernel_um,\n"
    "                det_threshold=det_threshold,\n"
    "                frame_index=i, window_size=W,\n"
    "            )\n"
    "            unet_feat = model._index_features(\n"
    "                unet_out[:, i], det_c, det_m,\n"
    "            )\n"
    "            frame_det.append((det_c, det_p, det_m, matches, unet_feat))\n"
    "\n"
    "            # Node recall:"
)

target.write_text(src)
assert "det_threshold: float = 0.3,\n) -> tuple[float, float, float]:" in src

In [14]:
lines = target.read_text().splitlines(keepends=True)

# line 1201 in your grep output is 1-indexed; list index is 1200
assert "Best score (acc" in lines[1200], f"unexpected content at index 1200: {lines[1200]!r}"

# metrics_block = (
#     '    metrics = {"best_score": best_score, "final_edge_loss": edge_loss,\n'
#     '               "final_det_loss": det_loss, "final_test_acc": test_acc,\n'
#     '               "final_test_recall": test_recall}\n'
# )

metrics_block = (
    '    metrics = {"best_score": best_score, "final_edge_loss": edge_loss,\n'
    '               "final_det_loss": det_loss, "final_test_acc": test_acc,\n'
    '               "final_test_recall": test_recall,\n'
    '               "best_test_acc": best_test_acc, "best_test_recall": best_test_recall}\n'
)


lines.insert(1201, metrics_block)  # insert right after the print line
target.write_text("".join(lines))
print("inserted")

inserted


In [15]:
REPO_DIR = ARTIFACTS_WORK / "repo"

target = REPO_DIR / "scripts" / "train_unet_transformer.py"  # same path as before
src = target.read_text()

old = (
    "        if is_best:\n"
    "            best_score = score\n"
)
new = (
    "        if is_best:\n"
    "            best_score = score\n"
    "            best_test_acc, best_test_recall = test_acc, test_recall\n"
)

#to check my text is well constructed with respect to the .py file
assert src.count(old) == 1, f"expected exactly 1 match, found {src.count(old)}"

src = src.replace(old, new)
target.write_text(src)

# verify against the file, not just the in-memory string
current = target.read_text()
assert "best_test_acc, best_test_recall = test_acc, test_recall" in current
print("confirmed on disk")

confirmed on disk


In [16]:
import inspect
from train_unet_transformer import train
print(inspect.getsource(train).count("best_test_acc"))  # should be >= 2 (assignment + dict entry)

3


/usr/local/lib/python3.12/dist-packages/dask/array/image.py:7: FutureWarning: `find_available_plugins` is deprecated since version 0.25 and will be removed in version 0.27. The plugin infrastructure of `skimage.io` is deprecated. Instead, use `imageio` or other I/O packages directly.
  from skimage.io import imread as sk_imread
/usr/lib/python3.12/importlib/__init__.py:90: FutureWarning: `reset_plugins` is deprecated since version 0.25 and will be removed in version 0.27. The plugin infrastructure of `skimage.io` is deprecated. Instead, use `imageio` or other I/O packages directly.
  return _bootstrap._gcd_import(name[level:], package, level)


## Run configuration

In [17]:
METHOD = "unet_transformer"

# --------------- Reading weights from the baseline model ---------------------------------------
# in the test set, the baseline model scored 0.8, let's see if we can improve this
# Model checkpoint (relative to the repo, or an absolute path to your own).
WEIGHTS = f"weights/{METHOD}/split_0/edge_predictor_best.pth"

## Sweep of set 2

#### Computing average spacing between cells from input datasets

In [18]:
import numpy as np
import polars as pl
from tracking_cellmot.io import open_dataset
from scipy.spatial import cKDTree

# pick a couple of representative training videos
# sample_stems = train_stems[:3]
sample_stems  = train_pool_stems # use this option if you want to use the whole training set

all_nn_dists = []
all_voxel_sizes = []

for stem in sample_stems: #moving trhough each movie (folder)
    ds = open_dataset(TRAINING_PATH / stem, normalize=False, require_tracks=True,
                       load_image=False, downsample=(1, 1, 1))  # no downsample — real physical units
    tracks = ds.tracks #gets the tracks
    voxel_size = ds.scale  # physical size per voxel, (Z, Y, X) as commented in competition documentation
    all_voxel_sizes.append(voxel_size)

    attrs = tracks.node_attrs(attr_keys=["t", "z", "y", "x"]) #this is a polars dataframe
    for t in attrs["t"].unique().to_list(): #unique temporal steps
        frame = attrs.filter(pl.col("t") == t) #taking only the frame t (Z,Y,X are obtained)
        coords = frame.select(["z", "y", "x"]).to_numpy() * np.array(voxel_size)  # -> getting coordinate and then physical microns
        if len(coords) < 2: #not a 3d space
            continue
        # nearest-neighbor distance for each node in this frame
        
        tree = cKDTree(coords)
        dists, _ = tree.query(coords, k=2)  # k=2: closest point to itself (0) + true nearest neighbor
        all_nn_dists.extend(dists[:, 1]) #to concantenate the array

all_nn_dists = np.array(all_nn_dists)
p10 = np.percentile(all_nn_dists,10)
median = np.median(all_nn_dists)
p90 = np.percentile(all_nn_dists,90)
print(f"voxel_size (Z,Y,X) per video: {all_voxel_sizes}")
print(f"nearest-neighbor distance (microns): "
      f"p10={p10:.2f}  "
      f"median={median:.2f}  "
      f"p90={p90:.2f}")

voxel_size (Z,Y,X) per video: [(1.625, 0.40625, 0.40625), (1.625, 0.40625, 0.40625), (1.625, 0.40625, 0.40625), (1.625, 0.40625, 0.40625), (1.625, 0.40625, 0.40625), (1.625, 0.40625, 0.40625), (1.625, 0.40625, 0.40625), (1.625, 0.40625, 0.40625), (1.625, 0.40625, 0.40625), (1.625, 0.40625, 0.40625), (1.625, 0.40625, 0.40625), (1.625, 0.40625, 0.40625), (1.625, 0.40625, 0.40625), (1.625, 0.40625, 0.40625), (1.625, 0.40625, 0.40625), (1.625, 0.40625, 0.40625), (1.625, 0.40625, 0.40625), (1.625, 0.40625, 0.40625), (1.625, 0.40625, 0.40625), (1.625, 0.40625, 0.40625), (1.625, 0.40625, 0.40625), (1.625, 0.40625, 0.40625), (1.625, 0.40625, 0.40625), (1.625, 0.40625, 0.40625), (1.625, 0.40625, 0.40625), (1.625, 0.40625, 0.40625), (1.625, 0.40625, 0.40625), (1.625, 0.40625, 0.40625), (1.625, 0.40625, 0.40625), (1.625, 0.40625, 0.40625), (1.625, 0.40625, 0.40625), (1.625, 0.40625, 0.40625), (1.625, 0.40625, 0.40625), (1.625, 0.40625, 0.40625), (1.625, 0.40625, 0.40625), (1.625, 0.40625, 0.40625

### Performing sweep

In [19]:
import itertools
import json
import time
from pathlib import Path
import gc
import torch
import random

from train_unet_transformer import train

RESULTS_PATH = Path("hp_search_results_part3.jsonl")
random.seed(0)
n_random_trials = 12  #approx 4 hours with 12 trials



def log_result(record: dict) -> None:
    with open(RESULTS_PATH, "a") as f:
        f.write(json.dumps(record) + "\n")

def run_trial(trial_id: str, **kwargs) -> dict:
    t0 = time.monotonic()
    try:
        model, metrics = train(**kwargs)
    finally:
        # ensure cleanup happens even if this trial itself OOMs
        if "model" in dir():
            del model
        gc.collect()
        torch.cuda.empty_cache()
    elapsed = time.monotonic() - t0
    record = {
        "trial_id": trial_id,
        "elapsed_s": round(elapsed, 1),
        **{k: (str(v) if isinstance(v, Path) else v) for k, v in kwargs.items()
           if k not in ("data_dir", "splits_file", "unet_layers", "full_checkpoint")},
        **metrics,
    }
    log_result(record)
    print(f"[{trial_id}] score={metrics['best_score']:.4f}  ({elapsed:.0f}s)")
    return record


FULL_CHECKPOINT = ARTIFACTS_SRC / "weights" / METHOD / "split_0" / "edge_predictor_best.pth"
assert FULL_CHECKPOINT.exists(), f"not found: {FULL_CHECKPOINT}"

# -----------------------------------------------------------------------
# Now we are going to fix the variables we found in the previous searches 
# -----------------------------------------------------------------------
LR = 0.00005
DET_LOSS_WEIGHT = 4
DET_NEG_WEIGHT = 0.010
DET_TRHESHOLD = 0.80

FIXED = dict(
    data_dir=Path(DATA_PATH)/"train",
    splits_file=ARTIFACTS_WORK/"kaggle_train_val_splits.json",
    fold=0,
    n_epochs=4,
    max_iters=200,
    seed=0,
    data_parallel=False, #set it to False given the small batch_size
    batch_size=2,
    method="hp_search_tier1",
    full_checkpoint=FULL_CHECKPOINT,   # <-- warm-start every trial from the 0.80 baseline
    lr = LR,
    det_loss_weight = DET_LOSS_WEIGHT,
    det_neg_weight = DET_NEG_WEIGHT,
    det_threshold = DET_TRHESHOLD,
    
)


# tier1b_space = {
#     "lr": [1e-5, 3e-5, 5e-5],              # bracket below and around the current winner
#     "det_loss_weight": [2, 3, 4],          # move araound the optimal = 3
#     "det_neg_weight": [0.003, 0.01, 0.03], #similar best score: 0.003 -> best score 0.908 and 0.03 -> best score 0.907
#     "det_threshold": [0.8, 0.92, 0.99],      # move from 0.80 to 0.99
# }

#-----------------------------------------------------------------
#Set 2: The variables are
#pool_kernel_um
#window_size
#these values are set according to the results from the mean distance between cells
#-----------------------------------------------------------------
MEAN_NODE_DISTANCE = median #um
tier2_space = {
    "pool_kernel_um":     [0.1*MEAN_NODE_DISTANCE,0.3*MEAN_NODE_DISTANCE,MEAN_NODE_DISTANCE,0.7*MEAN_NODE_DISTANCE],  # fill from d_nn above. This sets the radius of a local max-pooling window to decide around that radius what is the voxel with the largest probability of being a cell
                                  # if it is too large, multiple cells might be merged but too small will create artificial cells.
 #   "max_match_distance": [7],  # fill from d_nn above. It sets how fall apart a detected peak and a GT node are allowed to be and still count
                                  # as the same cell. it's purely about scoring/labeling those detections against the answer key.
                                # 7 um is the value defined in the competition documentation
    "window_size":        [2, 3], #temporal windows to consider parent-child nodes
}


keys = list(tier2_space.keys())
combos = list(itertools.product(*tier2_space.values()))
random.shuffle(combos)


#for i, combo in enumerate(itertools.product(*tier1_space.values())):
# for i, combo in enumerate(combos[:1]): # to test one sample
for i, combo in enumerate(combos[:n_random_trials]):
    kwargs = dict(zip(keys, combo))
    run_trial(f"tier2_{i:03d}", **FIXED, **kwargs)
    print(torch.cuda.memory_allocated() / 1e9, "GB allocated")
    print(torch.cuda.memory_reserved() / 1e9, "GB reserved")

Fold 0: 8 train, 2 test
Loading train (8 datasets)...


train: 100%|██████████| 8/8 [00:04<00:00,  1.69it/s]

  train done: 787 windows total
Loading test (2 datasets)...



test: 100%|██████████| 2/2 [00:00<00:00,  2.28it/s]

  test done: 164 windows total
max_nodes=17


...

Training:  75%|███████▌  | 3/4 [17:25<04:21, 261.59s/it, acc=0.9996, det=0.0064, edge=0.0003]

  Epoch   3/4 | edge=0.0003 | det=0.0064 | test_loss=0.0006 | acc=0.9996 | recall=0.8346 | best=0.8463   | train=238.8s test=22.3s


Training: 100%|██████████| 4/4 [17:25<00:00, 261.49s/it, acc=0.9996, det=0.0064, edge=0.0003]


Best score (acc*recall): 0.8463, saved to /kaggle/working/cellmot-baseline-artifacts/repo/weights/hp_search_tier1/split_0/edge_predictor_best.pth


[tier2_007] score=0.8463  (1052s)
0.019136512 GB allocated
0.245366784 GB reserved


In [20]:
import pandas as pd

df = pd.read_json("hp_search_results_part3.jsonl", lines=True)
df = df.sort_values("best_score", ascending=False)
print(df.head(10))

    trial_id  elapsed_s  fold  n_epochs  max_iters  seed  data_parallel  \
1  tier2_001     1586.8     0         4        200     0          False   
4  tier2_004     1086.9     0         4        200     0          False   
3  tier2_003     1068.8     0         4        200     0          False   
5  tier2_005     1552.9     0         4        200     0          False   
6  tier2_006     1516.3     0         4        200     0          False   
7  tier2_007     1051.7     0         4        200     0          False   
0  tier2_000     1056.2     0         4        200     0          False   
2  tier2_002     1521.1     0         4        200     0          False   

   batch_size           method       lr  ...  det_threshold  pool_kernel_um  \
1           2  hp_search_tier1  0.00005  ...            0.8        2.499346   
4           2  hp_search_tier1  0.00005  ...            0.8        2.499346   
3           2  hp_search_tier1  0.00005  ...            0.8        7.498037   
5       

In [21]:
# sanity-check marginal effect of each param independently
for param in ["lr", "det_loss_weight", "det_neg_weight", "det_threshold"]:
    print(df.groupby(param)["best_score"].agg(["mean", "std", "count"]))

             mean       std  count
lr                                
0.00005  0.865245  0.049197      8
                     mean       std  count
det_loss_weight                           
4                0.865245  0.049197      8
                    mean       std  count
det_neg_weight                           
0.01            0.865245  0.049197      8
                   mean       std  count
det_threshold                           
0.8            0.865245  0.049197      8


In [22]:
import subprocess
target = ARTIFACTS_WORK / "repo/scripts/train_unet_transformer.py"

result = subprocess.run(["grep", "-n", "max_match_distance\|def train_epoch\|def evaluate\|def train(", str(target)],
                         capture_output=True, text=True)
print(result.stdout)

627:    max_match_distance: float = 5.0,
647:    max_match_distance : maximum physical distance for detection→GT matching.
713:                if min_d[idx] > max_match_distance:
782:def train_epoch(
921:def evaluate(
1005:def train(



<>:4: SyntaxWarning: invalid escape sequence '\|'
<>:4: SyntaxWarning: invalid escape sequence '\|'
/tmp/ipykernel_22/1791499151.py:4: SyntaxWarning: invalid escape sequence '\|'
  result = subprocess.run(["grep", "-n", "max_match_distance\|def train_epoch\|def evaluate\|def train(", str(target)],
